# NewsScope - 05. Topic Modelling

## Phase F: Unsupervised Topic Discovery and Thematic Analysis

This notebook implements and evaluates latent topic discovery on the **AG News** dataset:
1. **Classical Topic Models (Stage B)**:
   - **Latent Dirichlet Allocation (LDA)**: Generative probabilistic model based on document-topic and topic-word Dirichlet distributions.
   - **Non-negative Matrix Factorization (NMF)**: Linear algebraic matrix factorisation ($V \approx W \times H$) decomposing TF-IDF term weights.
   - **Latent Semantic Analysis (LSA)**: Dimensionality reduction via Truncated Singular Value Decomposition (SVD).
2. **Neural Topic Modelling (Stage C - Pending Dependency Setup)**:
   - **BERTopic**: SentenceTransformer embeddings (`all-MiniLM-L6-v2`) + UMAP dimensionality reduction + HDBSCAN density clustering + c-TF-IDF keyword extraction.
3. **Evaluation & Thematic Validation (Stage D & E)**:
   - Topic coherence ($C_v$ and $C_{\text{NPMI}}$) and topic diversity.
   - Visualisation of top words, topic-document distributions, and post-hoc category alignment against AG News classes.

## Stage A: Setup & Data Sampling

We configure project directories, import core libraries, and draw a reproducible random sample of **3,000 documents** from `data/processed/train_processed.csv`.

### Unsupervised Integrity
- Ground-truth category labels (`label`, `category`) are strictly isolated upon data loading and never passed to vectorizers or topic models.
- The sample is drawn independently of category labels using fixed `random_state=42`.

In [1]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

# Set reproducible seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Directory paths
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "results" / "topic_modelling"
results_dir.mkdir(parents=True, exist_ok=True)

train_csv_path = processed_dir / "train_processed.csv"
assert train_csv_path.exists(), f"Processed data not found at: {train_csv_path}"

print("Setup completed successfully.")
print(f"  Scikit-Learn: {sklearn.__version__}")
print(f"  Pandas:       {pd.__version__}")
print(f"  NumPy:        {np.__version__}")
print(f"  Results dir:  {results_dir}")

Setup completed successfully.
  Scikit-Learn: 1.9.1
  Pandas:       3.1.1
  NumPy:        2.5.4
  Results dir:  C:\Users\Acer\Downloads\NewsScope\results\topic_modelling


### Load Processed Dataset & Extract Random Sample

We load `train_processed.csv`, extract a sample of 3,000 documents, and check for missing or empty text records.

In [2]:
# Load training dataset
full_df = pd.read_csv(train_csv_path)
print(f"Loaded full dataset: {len(full_df):,} articles.")

# Reproducible random sample of 3,000 documents (selected independently of labels)
sample_df = full_df.sample(n=3000, random_state=RANDOM_STATE).reset_index(drop=True)

# Extract document texts for topic modelling (unsupervised input)
documents = sample_df["clean_text"].tolist()

# Isolate category labels strictly for post-hoc evaluation
true_labels = sample_df["label"].to_numpy()
true_categories = sample_df["category"].to_numpy()
target_names = ["World", "Sports", "Business", "Sci/Tech"]

# Verification checks
print("\n=== Stage A Verification ===")
print(f"Sample size:               {len(documents):,} documents")
print(f"Null texts:                {sample_df['clean_text'].isna().sum()}")
print(f"Empty string texts:        {(sample_df['clean_text'].str.strip() == '').sum()}")
print("\nDistribution across categories in sample:")
display(sample_df["category"].value_counts().rename_axis("Category").reset_index(name="Count"))

print("\nPreview of sampled documents:")
display(sample_df[["category", "clean_text"]].head(4))

Loaded full dataset: 120,000 articles.

=== Stage A Verification ===
Sample size:               3,000 documents
Null texts:                0
Empty string texts:        0

Distribution across categories in sample:

Preview of sampled documents:


,Category,Count
0,Sports,794
1,Sci/Tech,757
2,World,736
3,Business,713


,category,clean_text
0,Business,bbc set major shake claim newspaper london bri...
1,Business,marsh averts cash crunch embattled insurance b...
2,Sports,jeter yankee look take control ap ap derek jet...
3,Sci/Tech,flying sun safety genesis capsule come back ea...


## Stage B: Classical Topic Modelling (LDA, NMF & LSA)

In this stage, we implement three classical topic modelling algorithms with $K = 4$ topics to compare against the four known thematic areas of AG News:
1. **Latent Dirichlet Allocation (LDA)**: A generative Bayesian model assuming each document is a mixture of latent topics, and each topic is a mixture of words (Dirichlet priors). Operates on discrete word count matrices (`CountVectorizer`).
2. **Non-negative Matrix Factorization (NMF)**: Decomposes the term-document matrix $V \approx W \times H$ under non-negativity constraints ($W, H \ge 0$). Matrix $W$ represents document-topic memberships, and matrix $H$ represents topic-word weights. Operates on TF-IDF term weights.
3. **Latent Semantic Analysis (LSA / TruncatedSVD)**: Decomposes the term-document matrix using Singular Value Decomposition ($V \approx U \Sigma V^T$) to uncover an orthogonal semantic vector space.

### Document Vectorization for Classical Models
We construct:
- **Count Matrix** (`CountVectorizer` with `min_df=3, max_df=0.95`) for LDA.
- **TF-IDF Matrix** (`TfidfVectorizer` with `min_df=3, max_df=0.95, norm='l2'`) for NMF and LSA.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Word counts for LDA
count_vec = CountVectorizer(min_df=3, max_df=0.95)
X_counts = count_vec.fit_transform(documents)
count_feature_names = count_vec.get_feature_names_out()

# 2. TF-IDF for NMF and LSA
tfidf_vec = TfidfVectorizer(min_df=3, max_df=0.95, norm="l2")
X_tfidf = tfidf_vec.fit_transform(documents)
tfidf_feature_names = tfidf_vec.get_feature_names_out()

print("=== Vectorization Summary ===")
print(f"Vocabulary size:               {len(count_feature_names):,} terms")
print(f"Count matrix shape (LDA):       {X_counts.shape}")
print(f"TF-IDF matrix shape (NMF/LSA): {X_tfidf.shape}")

=== Vectorization Summary ===
Vocabulary size:               4,201 terms
Count matrix shape (LDA):       (3000, 4201)
TF-IDF matrix shape (NMF/LSA): (3000, 4201)


### Helper Utilities for Topic Inspection
We define reusable functions to extract top words, format topic tables, and identify the most representative documents for each topic.

In [4]:
# Registry to store classical model results
classical_models = {}
classical_doc_topics = {}
classical_top_words = {}
classical_runtimes = {}

def get_top_topic_words(components, feature_names, top_n=10):
    """Extracts the top N words and their weights for each topic."""
    topics = {}
    for topic_idx, topic in enumerate(components):
        top_indices = topic.argsort()[:-top_n - 1:-1]
        topics[f"Topic {topic_idx + 1}"] = [feature_names[i] for i in top_indices]
    return pd.DataFrame(topics)

def display_representative_docs(doc_topics, documents, categories, model_name, n_docs=2):
    """Finds and displays documents with highest assignment probability for each topic."""
    print(f"\n--- Representative Documents for {model_name} ---")
    for topic_idx in range(doc_topics.shape[1]):
        top_doc_indices = doc_topics[:, topic_idx].argsort()[:-n_docs - 1:-1]
        print(f"\n[Topic {topic_idx + 1} - Top Assigned Documents]")
        for rank, doc_idx in enumerate(top_doc_indices):
            prob = doc_topics[doc_idx, topic_idx]
            cat = categories[doc_idx]
            text_snip = documents[doc_idx][:120]
            print(f"  Doc #{doc_idx} (Weight: {prob:.3f}, True Category: {cat}): {text_snip}...")

### 1. Latent Dirichlet Allocation (LDA)
We fit LDA with $K = 4$ topics using batch variational inference.

In [5]:
from sklearn.decomposition import LatentDirichletAllocation

print("Fitting Latent Dirichlet Allocation (LDA, K=4)...")
t0 = time.time()
lda_model = LatentDirichletAllocation(
    n_components=4,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=25,
)
lda_doc_topics = lda_model.fit_transform(X_counts)
lda_time = time.time() - t0

classical_models["LDA"] = lda_model
classical_doc_topics["LDA"] = lda_doc_topics
classical_runtimes["LDA"] = lda_time
classical_top_words["LDA"] = get_top_topic_words(lda_model.components_, count_feature_names)

print(f"LDA training completed in {lda_time:.2f}s")
print("\nTop 10 Words per Topic (LDA):")
display(classical_top_words["LDA"])

# Representative documents
display_representative_docs(lda_doc_topics, documents, true_categories, "LDA", n_docs=2)

Fitting Latent Dirichlet Allocation (LDA, K=4)...
LDA training completed in 38.46s

Top 10 Words per Topic (LDA):

--- Representative Documents for LDA ---

[Topic 1 - Top Assigned Documents]
  Doc #458 (Weight: 0.986, True Category: Business): cable veteran bid adelphia auction new york reuters several cable industry pioneer may bid part bankrupt cable operator ...
  Doc #1404 (Weight: 0.986, True Category: Sci/Tech): siebel creates cheaper smb software version siebel systemshas quietly introduced version crm customer relationship manag...

[Topic 2 - Top Assigned Documents]
  Doc #908 (Weight: 0.984, True Category: World): congress sends bush tax cut bill victory washington legislation extending three popular middle class tax cut rest decade...
  Doc #986 (Weight: 0.983, True Category: World): india pakistan discus joint project kashmir united state new york leader india pakistan yesterday hailed new era often f...

[Topic 3 - Top Assigned Documents]
  Doc #243 (Weight: 0.978, True C

,Topic 1,Topic 2,Topic 3,Topic 4
0,new,quot,said,game
1,reuters,president,iraq,ap
2,company,said,official,first
3,said,new,reuters,team
4,inc,bush,force,win
5,price,ap,killed,season
6,oil,say,attack,night
7,stock,talk,ap,last
8,microsoft,election,people,two
9,service,union,two,new


### 2. Non-negative Matrix Factorization (NMF)
We fit NMF with $K = 4$ topics using Non-negative Double Singular Value Decomposition (`nndsvd`) initialization.

In [6]:
from sklearn.decomposition import NMF

print("Fitting Non-negative Matrix Factorization (NMF, K=4)...")
t0 = time.time()
nmf_model = NMF(
    n_components=4,
    random_state=RANDOM_STATE,
    init="nndsvd",
    max_iter=300,
)
nmf_doc_topics = nmf_model.fit_transform(X_tfidf)
nmf_time = time.time() - t0

classical_models["NMF"] = nmf_model
classical_doc_topics["NMF"] = nmf_doc_topics
classical_runtimes["NMF"] = nmf_time
classical_top_words["NMF"] = get_top_topic_words(nmf_model.components_, tfidf_feature_names)

print(f"NMF training completed in {nmf_time:.2f}s")
print("\nTop 10 Words per Topic (NMF):")
display(classical_top_words["NMF"])

# Representative documents
display_representative_docs(nmf_doc_topics, documents, true_categories, "NMF", n_docs=2)

Fitting Non-negative Matrix Factorization (NMF, K=4)...
NMF training completed in 0.11s

Top 10 Words per Topic (NMF):

--- Representative Documents for NMF ---

[Topic 1 - Top Assigned Documents]
  Doc #223 (Weight: 0.155, True Category: Sports): umass arrest game alcs ap ap university massachusetts police arrested people early wednesday morning red sox tied yankee...
  Doc #2568 (Weight: 0.152, True Category: Sci/Tech): biotechnology shaking pet world ap ap biotechnology revolution shaking pet world...

[Topic 2 - Top Assigned Documents]
  Doc #1158 (Weight: 0.354, True Category: Business): stock future rise oil price ease new york reuters stock pointed slightly higher open monday investor seek bargain oil pr...
  Doc #63 (Weight: 0.330, True Category: Business): oil back winter stock reuters reuters oil price charged back barrel wednesday fresh fall heating oil stock stoked worry ...

[Topic 3 - Top Assigned Documents]
  Doc #757 (Weight: 0.176, True Category: Business): ibm sell co

,Topic 1,Topic 2,Topic 3,Topic 4
0,ap,oil,company,quot
1,game,price,microsoft,said
2,first,stock,new,iraq
3,team,reuters,inc,president
4,win,barrel,service,killed
5,season,york,google,official
6,night,crude,software,iraqi
7,league,high,business,election
8,victory,higher,computer,say
9,world,target,corp,afp


### 3. Latent Semantic Analysis (LSA / TruncatedSVD)
We fit TruncatedSVD with $K = 4$ components on the TF-IDF matrix.

In [7]:
from sklearn.decomposition import TruncatedSVD

print("Fitting Latent Semantic Analysis (LSA / TruncatedSVD, K=4)...")
t0 = time.time()
lsa_model = TruncatedSVD(
    n_components=4,
    random_state=RANDOM_STATE,
)
lsa_doc_topics = lsa_model.fit_transform(X_tfidf)
lsa_time = time.time() - t0

classical_models["LSA"] = lsa_model
classical_doc_topics["LSA"] = lsa_doc_topics
classical_runtimes["LSA"] = lsa_time
classical_top_words["LSA"] = get_top_topic_words(lsa_model.components_, tfidf_feature_names)

print(f"LSA training completed in {lsa_time:.2f}s")
print("\nTop 10 Words per Topic (LSA):")
display(classical_top_words["LSA"])

# Representative documents
display_representative_docs(lsa_doc_topics, documents, true_categories, "LSA", n_docs=2)

Fitting Latent Semantic Analysis (LSA / TruncatedSVD, K=4)...
LSA training completed in 0.03s

Top 10 Words per Topic (LSA):

--- Representative Documents for LSA ---

[Topic 1 - Top Assigned Documents]
  Doc #1158 (Weight: 0.280, True Category: Business): stock future rise oil price ease new york reuters stock pointed slightly higher open monday investor seek bargain oil pr...
  Doc #2580 (Weight: 0.265, True Category: World): stock turn higher oil hit record high new york stock enjoyed late day rally wednesday finish higher despite new record p...

[Topic 2 - Top Assigned Documents]
  Doc #223 (Weight: 0.243, True Category: Sports): umass arrest game alcs ap ap university massachusetts police arrested people early wednesday morning red sox tied yankee...
  Doc #2568 (Weight: 0.220, True Category: Sci/Tech): biotechnology shaking pet world ap ap biotechnology revolution shaking pet world...

[Topic 3 - Top Assigned Documents]
  Doc #63 (Weight: 0.343, True Category: Business): oil bac

,Topic 1,Topic 2,Topic 3,Topic 4
0,reuters,ap,oil,iraq
1,ap,game,ap,quot
2,new,quot,price,killed
3,said,team,reuters,iraqi
4,year,win,barrel,said
5,oil,two,game,official
6,quot,season,record,baghdad
7,stock,night,stock,troop
8,price,first,crude,oil
9,first,victory,season,president


### Comparison of Classical Topic Models
We compile the top words, runtime, and topic prevalence across all three classical models, and save the extracted topic words to `results/topic_modelling/topic_words.csv`.

In [8]:
# Compile consolidated topic words table
word_dfs = []
for model_name, df_words in classical_top_words.items():
    for col in df_words.columns:
        word_dfs.append({
            "Model": model_name,
            "Topic": col,
            "Top_Words": ", ".join(df_words[col].tolist())
        })

topic_words_df = pd.DataFrame(word_dfs)

# Save topic words to results directory
words_csv_path = results_dir / "topic_words.csv"
topic_words_df.to_csv(words_csv_path, index=False)
print(f"Saved topic words to: {words_csv_path}")

print("\n=== Classical Topic Models Summary ===")
display(topic_words_df)

# Document-topic distribution summary: proportion of documents primarily assigned to each topic
print("\n--- Document Distribution Across Topics ---")
dist_rows = []
for model_name, doc_topics in classical_doc_topics.items():
    dominant_topics = np.argmax(doc_topics, axis=1)
    counts = np.bincount(dominant_topics, minlength=4)
    dist_rows.append({
        "Model": model_name,
        "Runtime_s": round(classical_runtimes[model_name], 2),
        "Topic 1 Docs": counts[0],
        "Topic 2 Docs": counts[1],
        "Topic 3 Docs": counts[2],
        "Topic 4 Docs": counts[3],
    })

dist_df = pd.DataFrame(dist_rows)
display(dist_df)

Saved topic words to: C:\Users\Acer\Downloads\NewsScope\results\topic_modelling\topic_words.csv

=== Classical Topic Models Summary ===

--- Document Distribution Across Topics ---


,Model,Topic,Top_Words
0,LDA,Topic 1,"new, reuters, company, said, inc, price, oil, ..."
1,LDA,Topic 2,"quot, president, said, new, bush, ap, say, tal..."
2,LDA,Topic 3,"said, iraq, official, reuters, force, killed, ..."
3,LDA,Topic 4,"game, ap, first, team, win, season, night, las..."
4,NMF,Topic 1,"ap, game, first, team, win, season, night, lea..."
5,NMF,Topic 2,"oil, price, stock, reuters, barrel, york, crud..."
6,NMF,Topic 3,"company, microsoft, new, inc, service, google,..."
7,NMF,Topic 4,"quot, said, iraq, president, killed, official,..."
8,LSA,Topic 1,"reuters, ap, new, said, year, oil, quot, stock..."
9,LSA,Topic 2,"ap, game, quot, team, win, two, season, night,..."


,Model,Runtime_s,Topic 1 Docs,Topic 2 Docs,Topic 3 Docs,Topic 4 Docs
0,LDA,38.46,1030,660,494,816
1,NMF,0.11,836,270,999,895
2,LSA,0.03,2477,152,57,314


## Stage C: Neural Topic Modelling (BERTopic) - Pending Dependency Setup

*Status:* Stage C is preserved as a dedicated next stage. BERTopic requires `bertopic` and `sentence-transformers` (`all-MiniLM-L6-v2`), which will be configured once package installation is approved. 

The classical models in Stage B provide a solid baseline against which BERTopic's dense semantic embeddings and dynamic density-based topic discovery will be evaluated.